# RPT API — Interview Playground

## Setup

Before you start:
1. Copy `.env.example` to `.env`
2. Fill in the `RPT_TOKEN` you received
3. Run all cells from top to bottom

> **Note:** Do not commit `.env` to version control.

In [ ]:
import requests
import json
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, r2_score
from dotenv import load_dotenv
import os

load_dotenv(override=True)

API_TOKEN = os.getenv("RPT_TOKEN")
API_URL   = os.getenv("RPT_API_URL")

HEADERS = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json"
}

print("Token loaded:", bool(API_TOKEN))
print("API URL:", API_URL)

---

## Dataset: Montgomery County Employee Salaries

This dataset contains salary information for employees of Montgomery County, MD (USA), covering the year 2016. It is sourced from an open government data portal.

### Feature Descriptions

| Column | Type | Description |
|---|---|---|
| `gender` | categorical | Employee gender (`M` / `F`) |
| `department` | categorical | Short department code (e.g. `POL`, `FRS`) |
| `department_name` | categorical | Full department name |
| `division` | categorical | Sub-unit within the department |
| `assignment_category` | categorical | Employment type, e.g. `Fulltime-Regular`, `Parttime-Regular` |
| `employee_position_title` | categorical | Job title |
| `date_first_hired` | string | Date the employee was first hired (MM/DD/YYYY) |
| `year_first_hired` | numeric | Year extracted from `date_first_hired` |
| `current_annual_salary` | numeric | **Target column** — gross annual salary in USD |

In [ ]:
df = pd.read_csv("employee_salaries.csv")
print(df.shape)
df.head()

---

## Stakeholder Scenario

You receive the following message from an HR business partner:

> *"Hi, we've been getting questions from managers about whether salaries in our county are fair and consistent. I heard you can use that new AI model to get some predictions — could you take a look at our employee data and tell me what the model thinks? It would be great to have something concrete to show leadership by end of week."*

---

**Your task:** Use the RPT API to address this request.

There is intentionally no single correct answer. Think about:
- What is the actual prediction goal?
- Which columns should be used as context — and which ones shouldn't?
- How do you evaluate whether the model is useful?
- What would you tell the stakeholder based on your results?

---

## API Helpers

The RPT API takes a flat list of rows. Rows with known values serve as **context** (in-context examples). Rows with `"[PREDICT]"` in the target column are the **query rows** the model predicts.

The helpers below are provided as a starting point — feel free to modify them.

In [ ]:
def get_payload(context_df, eval_df_batch, target_col="current_annual_salary"):
    context_rows = json.loads(context_df.to_json(orient="records"))
    eval_copy = eval_df_batch.copy()
    eval_copy[target_col] = "[PREDICT]"
    eval_rows = json.loads(eval_copy.to_json(orient="records"))
    return {"rows": context_rows + eval_rows}

def get_response(payload):
    response = requests.post(API_URL, headers=HEADERS, json=payload, timeout=30)
    if not response.ok:
        print(f"Error {response.status_code}: {response.text}")
    response.raise_for_status()
    return response.json()

def get_predictions(result, target_col="current_annual_salary"):
    return pd.json_normalize(
        result["prediction"]["predictions"],
        record_path=target_col
    )["prediction"].astype(float).values

def evaluate(context_df, eval_df, target_col="current_annual_salary", batch_size=25):
    y_true_all, y_pred_all = [], []
    for start in range(0, len(eval_df), batch_size):
        batch = eval_df.iloc[start:start + batch_size]
        payload = get_payload(context_df, batch, target_col)
        result = get_response(payload)
        y_pred_all.extend(get_predictions(result, target_col))
        y_true_all.extend(batch[target_col].values)
    y_true = np.array(y_true_all)
    y_pred = np.array(y_pred_all)
    return {
        "MSE": mean_squared_error(y_true, y_pred),
        "R2":  r2_score(y_true, y_pred),
        "y_true": y_true,
        "y_pred": y_pred
    }

---

## Your Work

Use the cells below to explore the dataset and work with the API.